# CapacityIQ — Managed Services Workforce Forecasting

**Source data**
- `assign.xlsx` — 5,871 incident/ticket records (May 2024 – Jun 2026)
- `time_work.xlsx` — 87,393 timesheet rows across 3 sheets (May 2024 – May 2026)

**Structure**
| Section | Answers |
|---|---|
| 0. EDA | Exploratory analysis that informed the model design |
| Q1 | 13-week Prophet forecast — hours, FTE, tickets by tech/spec |
| Q2 | XGBoost + SHAP feature importance |
| Q3 | Demand overlay: projects, CRs, evergreening |
| Q4 | Seasonality, calendar effects, minimum staffing |
| Q5 | Capacity from timesheets — delivery vs non-delivery split |
| Q6 | Weekly demand vs capacity gaps, heatmap, resource alerts |
| Q7 | Minimum weekly refresh inputs |
| Q8 | Base / High / Low scenarios |

In [ ]:
# Uncomment and run once to install dependencies
# !pip install prophet xgboost shap openpyxl matplotlib seaborn plotly holidays

In [11]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

from prophet import Prophet
from xgboost import XGBRegressor
import shap
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

plt.rcParams.update({
    'figure.facecolor': '#080F1E',
    'axes.facecolor':   '#0F1E3A',
    'axes.edgecolor':   '#1E3058',
    'axes.labelcolor':  '#7A93B5',
    'xtick.color':      '#3D5070',
    'ytick.color':      '#3D5070',
    'text.color':       '#E8EEF7',
    'grid.color':       '#1E3058',
    'grid.alpha':        0.5,
    'figure.dpi':        130,
})
TEAL   = '#00C4A0'
AMBER  = '#F5A623'
RED    = '#F04F4F'
GREEN  = '#3DBC78'
TECH_COLORS = {
    'Dynamics 365 FO':                    '#00C4A0',
    'Dynamics 365 CE':                    '#4A9EFF',
    'Microsoft Azure':                    '#F5A623',
    'Microsoft Cloud Data Warehouse':     '#A855F7',
    'Dynamics NAV':                       '#3DBC78',
    'Power Platform':                     '#F472B6',
    'Dynamics 365 BC':                    '#22D3EE',
    'Dynamics 365 FSCM':                  '#F97316',
    'Microsoft Cloud Support Infrastructure': '#6B7280',
}

DATA_DIR = Path('.')  # change if files are elsewhere
print('Setup complete')

ImportError: Numba needs NumPy 2.3 or less. Got NumPy 2.4.

---
## Section 0 — Exploratory Data Analysis
Replicates the analysis from the initial data review.

In [ ]:
# ── Load assign.xlsx (incident / ticket data) ──────────────────────────────────
tickets = pd.read_excel(DATA_DIR / 'assign.xlsx', sheet_name='Page 1',
                        parse_dates=['Opened', 'Closed'])

print(f'Tickets shape: {tickets.shape}')
print(f'Date range — Opened: {tickets.Opened.min().date()} → {tickets.Opened.max().date()}')
print(f'Date range — Closed: {tickets.Closed.min().date()} → {tickets.Closed.max().date()}')
print(f'Columns: {list(tickets.columns)}')
tickets.head(3)

In [ ]:
# ── Load time_work.xlsx (timesheets, 3 pages) ──────────────────────────────────
xl = pd.ExcelFile(DATA_DIR / 'time_work.xlsx')
ts = pd.concat(
    [xl.parse(s, parse_dates=['Date']) for s in xl.sheet_names],
    ignore_index=True
)
ts['Date'] = pd.to_datetime(ts['Date'])
ts['week_start'] = ts['Date'] - pd.to_timedelta(ts['Date'].dt.dayofweek, unit='D')
ts['week_start'] = ts['week_start'].dt.normalize()
ts['ticket_type'] = ts['Number'].str[:3].fillna('UNK')

print(f'Timesheets shape: {ts.shape}')
print(f'Date range: {ts.Date.min().date()} → {ts.Date.max().date()}')
print(f'Total hours: {ts["Time worked"].sum():,.0f}')
print(f'Unique users: {ts.User.nunique()} | Companies: {ts.Company.nunique()} | Business services: {ts["Business service"].nunique()}')
ts.head(3)

In [ ]:
# ── Key distributions ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Technology split
tech_hrs = ts.groupby('Technology')['Time worked'].sum().sort_values(ascending=True)
colors_bar = [TECH_COLORS.get(t, TEAL) for t in tech_hrs.index]
axes[0].barh(tech_hrs.index, tech_hrs.values, color=colors_bar, edgecolor='none')
axes[0].set_title('Total hours by technology', fontsize=11)
axes[0].set_xlabel('Hours')

# Specialisation split (top 10)
spec_hrs = ts.groupby('Specialisation')['Time worked'].sum().nlargest(10).sort_values()
axes[1].barh(spec_hrs.index, spec_hrs.values, color=TEAL, alpha=0.8, edgecolor='none')
axes[1].set_title('Total hours by specialisation (top 10)', fontsize=11)

# Ticket type split
tt_hrs = ts.groupby('ticket_type')['Time worked'].sum().sort_values(ascending=False)
axes[2].bar(tt_hrs.index, tt_hrs.values,
            color=[TEAL, AMBER, GREEN, RED, '#A855F7', '#22D3EE'][:len(tt_hrs)],
            edgecolor='none')
axes[2].set_title('Hours by ticket type', fontsize=11)
axes[2].set_ylabel('Hours')

fig.tight_layout()
plt.show()

In [ ]:
# ── Weekly delivery trend by technology ───────────────────────────────────────
weekly_tech = (
    ts.groupby(['week_start', 'Technology'])['Time worked']
    .sum()
    .unstack('Technology')
    .fillna(0)
)
# Keep only techs with meaningful hours
main_techs = weekly_tech.columns[weekly_tech.mean() > 20]
weekly_tech = weekly_tech[main_techs]

fig, ax = plt.subplots(figsize=(14, 5))
for tech in main_techs:
    color = TECH_COLORS.get(tech, TEAL)
    ax.plot(weekly_tech.index, weekly_tech[tech],
            label=tech, color=color, linewidth=1.4, alpha=0.85)
    ax.fill_between(weekly_tech.index, weekly_tech[tech], alpha=0.06, color=color)

ax.set_title('Weekly hours by technology (historical)', fontsize=12)
ax.set_xlabel('Week')
ax.set_ylabel('Hours')
ax.legend(fontsize=8, loc='upper left', framealpha=0.1)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── GEN record classification (productive vs overhead) ────────────────────────
PRODUCTIVE_GEN = {
    'Roadmap Delivery', 'Evergreening', 'Release Management',
    'RACI – proactive work', 'Environment Management Activities',
    'Incidents', 'Incidents KSA', 'Incidents MAR', 'Incidents UAE',
    'Incidents RSA', 'ACCA Daily/weekly/monthly tasks', 'ACCA DT',
}
OVERHEAD_GEN = {
    'SDM / EM / EL', 'SDM / EM / EL (FO)', 'Pod Lead',
    'Daily Stand-ups', 'Client meeting', 'WBD internal meetings',
    'General admin',
}
INVESTMENT_GEN = {'Knowledge Transfer'}

def classify_row(row):
    if row['ticket_type'] in ('INC', 'CHG', 'SCT', 'PRB'):
        return 'Delivery'
    desc = str(row.get('Short description', ''))
    if desc in PRODUCTIVE_GEN:
        return 'Proactive Delivery'
    if desc in OVERHEAD_GEN:
        return 'Overhead'
    if desc in INVESTMENT_GEN:
        return 'Investment'
    if row['ticket_type'] == 'GEN':
        return 'GEN-Other'
    return 'Other'

ts['category'] = ts.apply(classify_row, axis=1)

cat_summary = ts.groupby('category')['Time worked'].agg(['sum','count'])
cat_summary['pct_hours'] = (cat_summary['sum'] / cat_summary['sum'].sum() * 100).round(1)
cat_summary.columns = ['Total Hours', 'Row Count', '% of Hours']
print(cat_summary.sort_values('Total Hours', ascending=False).to_string())

In [ ]:
# ── Seasonal patterns — Christmas and summer ──────────────────────────────────
ts['iso_week'] = ts['Date'].dt.isocalendar().week.astype(int)
ts['year']     = ts['Date'].dt.isocalendar().year.astype(int)

weekly_total = ts.groupby(['year', 'iso_week'])['Time worked'].sum().reset_index()
weekly_total['season'] = 'Normal'
weekly_total.loc[weekly_total['iso_week'].isin([51,52,1,2]), 'season'] = 'Christmas/NY'
weekly_total.loc[weekly_total['iso_week'].isin([28,29,30,31,32,33]), 'season'] = 'Summer'
weekly_total.loc[weekly_total['iso_week'].isin([16,17]), 'season'] = 'Easter'

season_stats = weekly_total.groupby('season')['Time worked'].agg(['mean','median','min','max','count'])
season_stats.columns = ['Mean hrs/wk', 'Median', 'Min', 'Max', 'Weeks']
print('Seasonal effort summary:\n')
print(season_stats.round(0).to_string())
print(f"\nNormal week avg: {weekly_total[weekly_total.season=='Normal']['Time worked'].mean():.0f}h")

# Christmas minimum staffing by technology
xmas = ts[ts['iso_week'].isin([51, 52, 1])]
xmas_by_tech = xmas.groupby('Technology')['Time worked'].agg(['sum','mean']).sort_values('sum', ascending=False)
xmas_by_tech.columns = ['Total xmas hrs', 'Avg hrs/entry']
print('\nChristmas period hours by technology:')
print(xmas_by_tech.round(1).to_string())

---
## Q1 — 13-Week Forecast: Hours, FTE, Ticket Volumes by Technology & Specialisation

**Approach:** Facebook Prophet with additive UK holiday effects and multiplicative yearly seasonality.  
One model per (Technology × Specialisation) cell. Ticket volume forecasted separately via a second Prophet model on weekly counts from `assign.xlsx`.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
UTILISATION_RATE = 0.85   # <-- edit here: 0.83, 0.80, etc.
HOURS_PER_WEEK   = 37.5   # standard contracted hours per FTE per week
FORECAST_WEEKS   = 13

FTE_DIVISOR = HOURS_PER_WEEK * UTILISATION_RATE
print(f'FTE divisor: {FTE_DIVISOR:.2f}h  (i.e. 1 FTE at {UTILISATION_RATE*100:.0f}% util = {FTE_DIVISOR:.1f}h/wk delivery)')

In [ ]:
# ── Prepare weekly delivery-only timeseries per Technology × Specialisation ───
# Use delivery + proactive delivery (not overhead/investment)
ts_delivery = ts[ts['category'].isin(['Delivery', 'Proactive Delivery'])].copy()

# Fill null Specialisation using the user's modal specialisation
user_modal_spec = (
    ts_delivery[ts_delivery['Specialisation'].notna()]
    .groupby('User')['Specialisation']
    .agg(lambda x: x.mode()[0] if len(x) else np.nan)
)
ts_delivery['Specialisation'] = ts_delivery.apply(
    lambda r: user_modal_spec.get(r['User'], r['Specialisation'])
              if pd.isna(r['Specialisation']) else r['Specialisation'],
    axis=1
)

# Aggregate to weekly
weekly = (
    ts_delivery
    .groupby(['week_start', 'Technology', 'Specialisation'])['Time worked']
    .sum()
    .reset_index()
    .rename(columns={'week_start': 'ds', 'Time worked': 'y'})
)
weekly = weekly[weekly['y'] > 0]

# Identify cells with enough history to train (≥ 26 weekly observations)
cell_counts = weekly.groupby(['Technology', 'Specialisation']).size()
valid_cells = cell_counts[cell_counts >= 26].reset_index()[['Technology', 'Specialisation']]
print(f'Valid (Technology, Specialisation) cells for Prophet: {len(valid_cells)}')
print(valid_cells.to_string(index=False))

In [ ]:
# ── UK bank holiday calendar for Prophet ──────────────────────────────────────
import holidays as hols

def build_uk_holidays_df(years=range(2024, 2028)):
    uk = hols.country_holidays('GB', subdiv='ENG')
    rows = []
    for y in years:
        for dt, name in hols.country_holidays('GB', subdiv='ENG', years=y).items():
            rows.append({'ds': pd.Timestamp(dt), 'holiday': name, 'lower_window': 0, 'upper_window': 1})
    return pd.DataFrame(rows).drop_duplicates('ds').sort_values('ds')

uk_holidays = build_uk_holidays_df()
print(f'UK holidays loaded: {len(uk_holidays)} entries')
print(uk_holidays[['ds','holiday']].head(10).to_string(index=False))

In [ ]:
# ── Train Prophet model per (Technology, Specialisation) cell ─────────────────
def train_cell_prophet(cell_df, holidays_df):
    """
    Trains a weekly Prophet model on a single Technology×Specialisation cell.
    Returns (model, forecast_df) or (None, None) if insufficient data.
    """
    df = cell_df[['ds', 'y']].copy().sort_values('ds')
    df = df.groupby('ds')['y'].sum().reset_index()  # ensure one row per week

    if len(df) < 26:
        return None, None

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,   # weekly granularity — weekly pattern N/A
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        interval_width=0.80,
        holidays=holidays_df,
        changepoint_prior_scale=0.05,    # conservative — managed services is stable
        seasonality_prior_scale=10.0,
    )
    # Suppress Prophet stdout
    import logging
    logging.getLogger('prophet').setLevel(logging.WARNING)
    logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

    m.fit(df)

    future = m.make_future_dataframe(periods=FORECAST_WEEKS, freq='W')
    forecast = m.predict(future)
    # Clip negative predictions
    forecast['yhat']       = forecast['yhat'].clip(lower=0)
    forecast['yhat_lower'] = forecast['yhat_lower'].clip(lower=0)
    forecast['yhat_upper'] = forecast['yhat_upper'].clip(lower=0)
    return m, forecast


# Train all valid cells
models    = {}   # (tech, spec) -> prophet model
forecasts = {}   # (tech, spec) -> forecast dataframe

for _, row in valid_cells.iterrows():
    key  = (row['Technology'], row['Specialisation'])
    cell = weekly[(weekly['Technology'] == key[0]) & (weekly['Specialisation'] == key[1])]
    m, fc = train_cell_prophet(cell, uk_holidays)
    if m is not None:
        models[key]    = m
        forecasts[key] = fc
        print(f'  ✓ {key[0]} / {key[1]}')

print(f'\nModels trained: {len(models)}')

In [ ]:
# ── Build 13-week forecast table ──────────────────────────────────────────────
forecast_start = pd.Timestamp.now().normalize() + pd.offsets.Week(weekday=0)  # next Monday
forecast_weeks = pd.date_range(forecast_start, periods=FORECAST_WEEKS, freq='W-MON')

records = []
for (tech, spec), fc in forecasts.items():
    fc_fwd = fc[fc['ds'] >= forecast_start].head(FORECAST_WEEKS).copy()
    fc_fwd['Technology']    = tech
    fc_fwd['Specialisation'] = spec
    fc_fwd['forecast_hours'] = fc_fwd['yhat'].round(1)
    fc_fwd['hours_low']      = fc_fwd['yhat_lower'].round(1)
    fc_fwd['hours_high']     = fc_fwd['yhat_upper'].round(1)
    fc_fwd['fte_demand']     = (fc_fwd['forecast_hours'] / FTE_DIVISOR).round(2)
    fc_fwd['week_num']       = range(1, len(fc_fwd) + 1)
    records.append(fc_fwd[['ds','week_num','Technology','Specialisation',
                             'forecast_hours','hours_low','hours_high','fte_demand']])

q1_forecast = pd.concat(records, ignore_index=True)
q1_forecast['week_label'] = 'Wk' + q1_forecast['week_num'].astype(str) + ' (' + q1_forecast['ds'].dt.strftime('%d %b') + ')'

# Summary pivot: hours by week × technology
pivot_hours = q1_forecast.pivot_table(
    index='week_label', columns='Technology', values='forecast_hours', aggfunc='sum'
).round(0)
pivot_hours.index.name = 'Week'

# Reorder by week number
pivot_hours['_sort'] = q1_forecast.drop_duplicates('week_label').set_index('week_label')['week_num']
pivot_hours = pivot_hours.sort_values('_sort').drop(columns='_sort')

print(f'Q1 Forecast — hours by week × technology (next {FORECAST_WEEKS} weeks):\n')
print(pivot_hours.to_string())

In [ ]:
# ── FTE demand pivot ──────────────────────────────────────────────────────────
pivot_fte = q1_forecast.pivot_table(
    index='week_label', columns='Technology', values='fte_demand', aggfunc='sum'
).round(1)
pivot_fte['_sort'] = q1_forecast.drop_duplicates('week_label').set_index('week_label')['week_num']
pivot_fte = pivot_fte.sort_values('_sort').drop(columns='_sort')
pivot_fte.index.name = 'Week'
print(f'Q1 Forecast — FTE demand by week × technology ({UTILISATION_RATE*100:.0f}% utilisation):\n')
print(pivot_fte.to_string())

In [ ]:
# ── Ticket volume forecast from assign.xlsx ───────────────────────────────────
tickets['week_start'] = tickets['Opened'] - pd.to_timedelta(tickets['Opened'].dt.dayofweek, unit='D')
tickets['week_start'] = tickets['week_start'].dt.normalize()

# Map assignment group to Technology
GROUP_TECH_MAP = {
    'FO': 'Dynamics 365 FO',
    'CE': 'Dynamics 365 CE',
    'NAV and BC': 'Dynamics NAV',
    'Power Platform': 'Power Platform',
    'Data': 'Microsoft Cloud Data Warehouse',
    'Azure': 'Microsoft Azure',
}
def map_group_to_tech(group):
    if pd.isna(group): return 'Other'
    for key, val in GROUP_TECH_MAP.items():
        if key.lower() in str(group).lower():
            return val
    return 'Other'

tickets['Technology'] = tickets['Assignment group'].apply(map_group_to_tech)

# Weekly counts by type and technology
weekly_tix = (
    tickets
    .groupby(['week_start', 'Technology', 'Category'])
    .size()
    .reset_index(name='count')
)

# Train Prophet on total weekly INC count per technology
def forecast_ticket_volume(weekly_tix, tech, category='Bug', horizon=FORECAST_WEEKS):
    sub = weekly_tix[(weekly_tix['Technology'] == tech)].groupby('week_start')['count'].sum().reset_index()
    sub.columns = ['ds', 'y']
    sub = sub[sub['y'] > 0]
    if len(sub) < 10:
        return None
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, interval_width=0.80,
                seasonality_mode='multiplicative', holidays=uk_holidays)
    import logging; logging.getLogger('prophet').setLevel(logging.WARNING)
    m.fit(sub)
    future = m.make_future_dataframe(periods=horizon, freq='W')
    fc = m.predict(future)
    fc['yhat'] = fc['yhat'].clip(lower=0).round(0)
    return fc[fc['ds'] >= forecast_start][['ds','yhat','yhat_lower','yhat_upper']].head(horizon)

# Forecast INC volumes for main techs
ticket_forecasts = {}
for tech in ['Dynamics 365 FO', 'Dynamics 365 CE', 'Microsoft Azure']:
    fc = forecast_ticket_volume(weekly_tix, tech)
    if fc is not None:
        ticket_forecasts[tech] = fc
        print(f'Ticket forecast trained: {tech}')

if ticket_forecasts:
    sample_tech = list(ticket_forecasts.keys())[0]
    print(f'\nSample — {sample_tech} weekly ticket volume forecast:')
    tf = ticket_forecasts[sample_tech].copy()
    tf['week'] = range(1, len(tf)+1)
    print(tf[['week','yhat','yhat_lower','yhat_upper']].rename(
        columns={'yhat':'Base','yhat_lower':'Low','yhat_upper':'High'}).to_string(index=False))

In [ ]:
# ── Plot forecast for dominant technology ─────────────────────────────────────
fo_cells = [(tech, spec) for tech, spec in forecasts if tech == 'Dynamics 365 FO']

fig, axes = plt.subplots(len(fo_cells), 1, figsize=(13, 3 * len(fo_cells)), sharex=True)
if len(fo_cells) == 1:
    axes = [axes]

for ax, (tech, spec) in zip(axes, fo_cells):
    fc  = forecasts[(tech, spec)]
    hist = weekly[(weekly['Technology'] == tech) & (weekly['Specialisation'] == spec)]
    color = TECH_COLORS.get(tech, TEAL)

    # Historical
    ax.plot(hist['ds'], hist['y'], color=color, alpha=0.6, linewidth=1.2, label='Actual')

    # Forecast window
    fc_fwd = fc[fc['ds'] >= forecast_start].head(FORECAST_WEEKS)
    ax.plot(fc_fwd['ds'], fc_fwd['yhat'], color=color, linewidth=1.8, linestyle='--', label='Forecast')
    ax.fill_between(fc_fwd['ds'], fc_fwd['yhat_lower'], fc_fwd['yhat_upper'],
                    color=color, alpha=0.15, label='80% CI')

    ax.axvline(forecast_start, color=AMBER, linewidth=0.8, linestyle=':', alpha=0.7)
    ax.set_title(f'{tech} — {spec}', fontsize=10)
    ax.set_ylabel('Hours/wk')
    ax.legend(fontsize=8, loc='upper left', framealpha=0.1)
    ax.grid(True, alpha=0.2)

fig.suptitle('Q1: 13-week Prophet forecast — D365 FO', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## Q2 — Feature Importance: Which Variables Best Explain Delivery Effort?

**Approach:** XGBoost regression on a weekly feature matrix. SHAP (SHapley Additive exPlanations) values identify which features push each prediction up or down. The top features from SHAP become the regressors fed into Prophet.

In [ ]:
# ── Build weekly feature matrix ───────────────────────────────────────────────
# Target: total delivery hours per week (all techs combined)
target_weekly = (
    ts[ts['category'].isin(['Delivery', 'Proactive Delivery'])]
    .groupby('week_start')['Time worked']
    .sum()
    .reset_index()
    .rename(columns={'week_start': 'week', 'Time worked': 'total_hours'})
    .sort_values('week')
)

# Weekly ticket metrics from assign.xlsx
weekly_tix_agg = tickets.copy()
weekly_tix_agg['is_high'] = weekly_tix_agg['Priority'].isin(['1 - Critical', '2 - High'])
tix_features = (
    weekly_tix_agg
    .groupby('week_start')
    .agg(
        new_tickets   = ('Number',  'count'),
        pct_high_pri  = ('is_high',  'mean'),
        chg_count     = ('Category', lambda x: (x == 'Change Request').sum()),
    )
    .reset_index()
    .rename(columns={'week_start': 'week'})
)

# Weekly open backlog (tickets opened but not yet closed)
backlog = []
for week in target_weekly['week']:
    open_count = ((tickets['Opened'] <= week) &
                  (tickets['Closed'].isna() | (tickets['Closed'] > week))).sum()
    backlog.append({'week': week, 'open_backlog': open_count})
backlog_df = pd.DataFrame(backlog)

# Merge everything
features = (
    target_weekly
    .merge(tix_features, on='week', how='left')
    .merge(backlog_df,   on='week', how='left')
    .sort_values('week')
)

# Lag features (most important for managed services — strong autocorrelation)
features['lag_1']      = features['total_hours'].shift(1)
features['lag_2']      = features['total_hours'].shift(2)
features['lag_4']      = features['total_hours'].shift(4)
features['rolling_4']  = features['total_hours'].shift(1).rolling(4).mean()
features['rolling_8']  = features['total_hours'].shift(1).rolling(8).mean()

# Calendar features
features['iso_week']   = features['week'].dt.isocalendar().week.astype(int)
features['month']      = features['week'].dt.month
features['is_xmas']    = features['iso_week'].isin([51, 52, 1, 2]).astype(int)
features['is_summer']  = features['iso_week'].between(28, 33).astype(int)
features['is_easter']  = features['iso_week'].isin([16, 17]).astype(int)

features = features.dropna().reset_index(drop=True)
print(f'Feature matrix shape: {features.shape}')
print(f'Columns: {list(features.columns)}')

In [ ]:
# ── Train XGBoost with time-series cross-validation ───────────────────────────
FEATURE_COLS = [
    'lag_1', 'lag_2', 'lag_4', 'rolling_4', 'rolling_8',
    'new_tickets', 'pct_high_pri', 'chg_count', 'open_backlog',
    'iso_week', 'month', 'is_xmas', 'is_summer', 'is_easter',
]

X = features[FEATURE_COLS].values
y = features['total_hours'].values

# Time-series split (no future leakage)
tscv = TimeSeriesSplit(n_splits=4)
mae_scores, r2_scores = [], []

xgb = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)

for train_idx, val_idx in tscv.split(X):
    xgb.fit(X[train_idx], y[train_idx])
    pred = xgb.predict(X[val_idx])
    mae_scores.append(mean_absolute_error(y[val_idx], pred))
    r2_scores.append(r2_score(y[val_idx], pred))

print(f'XGBoost cross-validation (TimeSeriesSplit, n=4)')
print(f'  MAE: {np.mean(mae_scores):.1f}h  (±{np.std(mae_scores):.1f}h)')
print(f'  R²:  {np.mean(r2_scores):.3f}  (±{np.std(r2_scores):.3f})')

# Refit on full dataset for SHAP
xgb.fit(X, y)
print('\nFull model trained for SHAP analysis')

In [ ]:
# ── SHAP feature importance ───────────────────────────────────────────────────
explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean absolute SHAP bar chart
mean_shap = np.abs(shap_values).mean(axis=0)
order     = np.argsort(mean_shap)[::-1]
axes[0].barh([FEATURE_COLS[i] for i in order[::-1]],
             mean_shap[order[::-1]],
             color=TEAL, edgecolor='none', alpha=0.85)
axes[0].set_title('Mean |SHAP| — global feature importance', fontsize=10)
axes[0].set_xlabel('Mean absolute SHAP value (hours impact)')
axes[0].grid(True, axis='x', alpha=0.3)

# SHAP beeswarm (requires shap's own figure)
plt.sca(axes[1])
shap.summary_plot(shap_values, features[FEATURE_COLS],
                  feature_names=FEATURE_COLS, show=False, plot_size=None,
                  color_bar=False, alpha=0.6)
axes[1].set_title('SHAP beeswarm — direction and magnitude', fontsize=10)

fig.tight_layout()
plt.show()

print('\nTop features to use as Prophet regressors (by mean |SHAP|):')
for i in order[:6]:
    print(f'  {FEATURE_COLS[i]:20s}  mean |SHAP| = {mean_shap[i]:.1f}h')

In [ ]:
# ── SHAP waterfall for a single week (most recent) ────────────────────────────
last_idx = len(X) - 1
shap.waterfall_plot(
    shap.Explanation(
        values       = shap_values[last_idx],
        base_values  = explainer.expected_value,
        data         = X[last_idx],
        feature_names= FEATURE_COLS,
    ),
    max_display=10,
)
plt.title(f'SHAP waterfall — week of {features.iloc[last_idx]["week"].date()}', fontsize=10)
plt.tight_layout()
plt.show()

---
## Q3 — Demand Overlay: Projects, Evergreening, Transitions, Large CRs

A simple input DataFrame that gets merged on top of the BAU Prophet forecast before gap analysis. Edit the `demand_overlay` table each week to add or remove demand.

In [ ]:
# ── Demand overlay input template ─────────────────────────────────────────────
# Edit this each week. week_start should be a Monday.
demand_overlay = pd.DataFrame([
    # week_start          technology              specialisation   type            company_project  hours
    dict(week_start='2026-06-09', technology='Dynamics 365 FO',  specialisation='FO Functional',  type='Transition',  project='Project Alpha',   hours=40),
    dict(week_start='2026-06-09', technology='Dynamics 365 FO',  specialisation='FO Technical',   type='Transition',  project='Project Alpha',   hours=16),
    dict(week_start='2026-06-16', technology='Dynamics 365 FO',  specialisation='FO Functional',  type='Transition',  project='Project Alpha',   hours=40),
    dict(week_start='2026-06-09', technology='Dynamics 365 CE',  specialisation='CE Functional',  type='Evergreening',project='Quarterly EG',    hours=20),
    dict(week_start='2026-06-23', technology='Microsoft Azure',  specialisation='Azure Integration',type='LargeCR',   project='Cloud Migration', hours=32),
])
demand_overlay['week_start'] = pd.to_datetime(demand_overlay['week_start'])

print('Demand overlay:')
print(demand_overlay.to_string(index=False))

In [ ]:
# ── Merge overlay with BAU forecast ──────────────────────────────────────────
# Aggregate overlay by week × tech × spec
overlay_agg = (
    demand_overlay
    .groupby(['week_start', 'technology', 'specialisation'])['hours']
    .sum()
    .reset_index()
    .rename(columns={'technology':'Technology','specialisation':'Specialisation','hours':'overlay_hours'})
)

# Merge into q1_forecast
q1_with_overlay = q1_forecast.merge(
    overlay_agg.rename(columns={'week_start':'ds'}),
    on=['ds','Technology','Specialisation'],
    how='left'
)
q1_with_overlay['overlay_hours'] = q1_with_overlay['overlay_hours'].fillna(0)
q1_with_overlay['total_hours']   = q1_with_overlay['forecast_hours'] + q1_with_overlay['overlay_hours']
q1_with_overlay['total_fte']     = (q1_with_overlay['total_hours'] / FTE_DIVISOR).round(2)

# Show weeks with overlay impact
overlay_weeks = q1_with_overlay[q1_with_overlay['overlay_hours'] > 0]
print('Weeks with overlay demand added:')
print(overlay_weeks[['week_label','Technology','Specialisation','forecast_hours','overlay_hours','total_hours','total_fte']]
      .to_string(index=False))

---
## Q4 — Seasonality, Calendar Effects, and Minimum Staffing Requirements

In [ ]:
# ── Prophet seasonality decomposition for FO Functional ───────────────────────
key = ('Dynamics 365 FO', 'FO Functional')
if key in models:
    m  = models[key]
    fc = forecasts[key]
    fig = m.plot_components(fc)
    fig.suptitle(f'Prophet components — {key[0]} / {key[1]}', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Seasonal multiplier derivation (empirical from your data) ─────────────────
weekly_all = (
    ts[ts['category'].isin(['Delivery','Proactive Delivery'])]
    .groupby(['week_start','iso_week'])['Time worked']
    .sum()
    .reset_index()
)
normal_avg = weekly_all[~weekly_all['iso_week'].isin([1,2,51,52,16,17,28,29,30,31,32,33])]['Time worked'].mean()
seasonal_mult = (
    weekly_all.groupby('iso_week')['Time worked']
    .mean()
    .rename('avg_hours')
    .to_frame()
)
seasonal_mult['seasonal_index'] = (seasonal_mult['avg_hours'] / normal_avg).round(3)

fig, ax = plt.subplots(figsize=(13, 4))
colors = [RED if x < 0.7 else AMBER if x < 0.92 else TEAL
          for x in seasonal_mult['seasonal_index']]
ax.bar(seasonal_mult.index, seasonal_mult['seasonal_index'], color=colors, edgecolor='none', alpha=0.85)
ax.axhline(1.0, color=TEAL, linestyle='--', linewidth=0.8, alpha=0.7, label='Baseline')
ax.axhline(0.85, color=AMBER, linestyle=':', linewidth=0.8, alpha=0.6, label='−15% threshold')
ax.set_xlabel('ISO week number')
ax.set_ylabel('Seasonal index')
ax.set_title('Q4: Empirical seasonal index by ISO week (derived from 2 years of timesheets)', fontsize=11)
ax.legend(fontsize=9, framealpha=0.1)
ax.grid(True, axis='y', alpha=0.25)
ax.set_xticks(range(1, 53, 2))
plt.tight_layout()
plt.show()

print('Lowest weeks (minimum staffing periods):')
print(seasonal_mult.nsmallest(8,'seasonal_index').to_string())

In [ ]:
# ── Minimum staffing: Christmas week by technology and grade ──────────────────
xmas_ts = ts[(ts['iso_week'].isin([51,52,1])) & (ts['category'].isin(['Delivery','Proactive Delivery']))]

xmas_min = (
    xmas_ts.groupby(['Technology','Specialisation'])
    .agg(
        total_hours   = ('Time worked', 'sum'),
        avg_wk_hours  = ('Time worked', lambda x: x.groupby(xmas_ts.loc[x.index,'week_start']).sum().mean()),
        min_wk_hours  = ('Time worked', lambda x: x.groupby(xmas_ts.loc[x.index,'week_start']).sum().min()),
        min_fte_equiv = ('Time worked', lambda x: x.groupby(xmas_ts.loc[x.index,'week_start']).sum().min() / FTE_DIVISOR),
    )
    .reset_index()
    .sort_values('avg_wk_hours', ascending=False)
)

print('Q4: Minimum staffing (Christmas weeks 51/52/1) by Technology × Specialisation:')
print(xmas_min.round(1).to_string(index=False))

---
## Q5 — Capacity Calculation from Timesheets

Separates productive delivery hours from leave, meetings, admin, and training.  
**Note:** Leave/absence is not present in these files — add a headcount × leave% adjustment manually or connect an HR data export.

In [ ]:
# ── Weekly capacity breakdown by category ─────────────────────────────────────
capacity_weekly = (
    ts.groupby(['week_start','category'])['Time worked']
    .sum()
    .unstack('category')
    .fillna(0)
    .sort_index()
)

# Delivery = Delivery + Proactive Delivery
capacity_weekly['Total Delivery'] = (
    capacity_weekly.get('Delivery', 0) +
    capacity_weekly.get('Proactive Delivery', 0)
)
capacity_weekly['Total Logged']   = capacity_weekly.sum(axis=1)
capacity_weekly['Utilisation %']  = (
    capacity_weekly['Total Delivery'] / capacity_weekly['Total Logged'] * 100
).round(1)

print('Last 8 weeks — capacity breakdown:')
print(capacity_weekly.tail(8).round(1).to_string())

In [ ]:
# ── Historical utilisation rate (delivery vs total logged) ────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# Stacked area: hours by category
plot_cats = [c for c in ['Delivery','Proactive Delivery','Overhead','Investment','GEN-Other','Other']
             if c in capacity_weekly.columns]
cat_colors = [TEAL,'#00897B',AMBER,'#A855F7','#6B7280','#374151']
axes[0].stackplot(
    capacity_weekly.index,
    [capacity_weekly.get(c, 0) for c in plot_cats],
    labels=plot_cats,
    colors=cat_colors[:len(plot_cats)],
    alpha=0.82,
)
axes[0].set_title('Q5: Weekly hours by activity type', fontsize=11)
axes[0].set_ylabel('Hours')
axes[0].legend(fontsize=8, loc='upper left', framealpha=0.1)
axes[0].grid(True, axis='y', alpha=0.2)

# Utilisation rate
axes[1].plot(capacity_weekly.index, capacity_weekly['Utilisation %'],
             color=TEAL, linewidth=1.5)
axes[1].fill_between(capacity_weekly.index, capacity_weekly['Utilisation %'],
                     alpha=0.12, color=TEAL)
axes[1].axhline(85, color=AMBER, linestyle='--', linewidth=0.9, label='85% target')
axes[1].set_title('Effective delivery utilisation %', fontsize=11)
axes[1].set_ylabel('%')
axes[1].set_ylim(0, 100)
axes[1].legend(fontsize=9, framealpha=0.1)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

recent = capacity_weekly.tail(13)
print(f'Recent 13-week avg utilisation: {recent["Utilisation %"].mean():.1f}%')
print(f'Recent 13-week avg delivery hours/wk: {recent["Total Delivery"].mean():.0f}h')

In [ ]:
# ── Capacity per technology based on headcount proxy ─────────────────────────
# Active users in last 3 months = proxy headcount
cutoff_3m = ts['week_start'].max() - pd.DateOffset(weeks=13)
active_hc = (
    ts[ts['week_start'] >= cutoff_3m]
    .groupby('Technology')['User']
    .nunique()
    .rename('active_users')
    .reset_index()
)
active_hc['weekly_capacity_hrs'] = active_hc['active_users'] * HOURS_PER_WEEK * UTILISATION_RATE
active_hc['weekly_capacity_fte'] = active_hc['active_users'] * UTILISATION_RATE

# Compare with recent actual delivery
recent_delivery_by_tech = (
    ts[(ts['week_start'] >= cutoff_3m) & ts['category'].isin(['Delivery','Proactive Delivery'])]
    .groupby('Technology')['Time worked']
    .mean()  # avg per log entry, need weekly
)
recent_wk_by_tech = (
    ts[(ts['week_start'] >= cutoff_3m) & ts['category'].isin(['Delivery','Proactive Delivery'])]
    .groupby(['week_start','Technology'])['Time worked']
    .sum()
    .groupby('Technology')
    .mean()
    .rename('avg_wk_delivery_hrs')
    .reset_index()
)

capacity_summary = active_hc.merge(recent_wk_by_tech, on='Technology', how='left')
capacity_summary['utilisation_vs_capacity'] = (
    capacity_summary['avg_wk_delivery_hrs'] / capacity_summary['weekly_capacity_hrs'] * 100
).round(1)

print(f'Q5: Capacity table ({UTILISATION_RATE*100:.0f}% utilisation, {HOURS_PER_WEEK}h/wk):\n')
print(capacity_summary.sort_values('active_users', ascending=False).round(1).to_string(index=False))

---
## Q6 — Weekly Demand vs Capacity Gaps for 13 Weeks by Technology, Specialisation, Skillset

In [ ]:
# ── Build per-technology headcount table (edit these numbers as staff change) ──
headcount = {
    'Dynamics 365 FO':                 103,
    'Dynamics 365 CE':                  50,
    'Microsoft Azure':                  36,
    'Microsoft Cloud Data Warehouse':   16,
    'Dynamics NAV':                     14,
    'Power Platform':                   13,
    'Dynamics 365 BC':                  12,
}

def calc_capacity(tech, util=UTILISATION_RATE, hpw=HOURS_PER_WEEK):
    hc = headcount.get(tech, 0)
    return hc * hpw * util

# Compute gap for each row in q1_with_overlay
q1_gap = q1_with_overlay.copy()
q1_gap['capacity_hours'] = q1_gap['Technology'].map(calc_capacity)
q1_gap['gap_hours']      = q1_gap['total_hours']    - q1_gap['capacity_hours']
q1_gap['gap_fte']        = q1_gap['total_fte']      - (q1_gap['capacity_hours'] / FTE_DIVISOR)
q1_gap['gap_pct']        = (q1_gap['gap_hours'] / q1_gap['capacity_hours'] * 100).round(1)
q1_gap['status']         = pd.cut(
    q1_gap['gap_pct'],
    bins=[-np.inf, 0, 10, 25, np.inf],
    labels=['OK', 'WATCH', 'AMBER', 'RED']
)

# Summary gap table
gap_pivot = q1_gap.pivot_table(
    index='week_label', columns='Technology', values='gap_pct', aggfunc='sum'
).round(1)
gap_pivot['_sort'] = q1_gap.drop_duplicates('week_label').set_index('week_label')['week_num']
gap_pivot = gap_pivot.sort_values('_sort').drop(columns='_sort')

print('Q6: Demand vs Capacity gap % by week × technology (positive = over capacity):\n')
print(gap_pivot.to_string())

In [ ]:
# ── Gap heatmap ───────────────────────────────────────────────────────────────
tech_cols = [c for c in gap_pivot.columns if c in headcount]
hm_data   = gap_pivot[tech_cols].fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))

cmap = mcolors.LinearSegmentedColormap.from_list(
    'capacity', ['#1a4a2a', '#1a3a2a', '#0F1E3A', '#5a3800', '#8a0000'],
    N=256
)

sns.heatmap(
    hm_data.T,
    ax=ax,
    cmap=cmap,
    center=0,
    vmin=-30, vmax=40,
    annot=True,
    fmt='.0f',
    annot_kws={'size': 8},
    linewidths=0.3,
    linecolor='#0F1E3A',
    cbar_kws={'label': 'Gap %  (+ = over capacity)', 'shrink': 0.6},
)
ax.set_title('Q6: Capacity gap heatmap (%) — next 13 weeks', fontsize=12)
ax.set_xlabel('Week')
ax.set_ylabel('Technology')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Automated resource alert table ────────────────────────────────────────────
alerts = q1_gap[q1_gap['gap_fte'] >= 0.5].copy()

GRADE_HINTS = {
    'FO Functional':       'C1/C2 Functional Consultant',
    'FO Technical':        'D1/D2 Technical Consultant',
    'CE Technical':        'D1/D2 Technical Consultant',
    'CE Functional':       'C1/C2 Functional Consultant',
    'Azure Integration':   'D1/D2 Azure Integration Engineer',
    'Azure Infrastructure':'D1/D2 Azure Infrastructure Engineer',
    'Data':                'D1/D2 Data Engineer',
    'SDM':                 'C2/B1 Service Delivery Manager',
}

alert_rows = []
for _, row in alerts.sort_values(['week_num','gap_fte'], ascending=[True,False]).iterrows():
    engage_by = row['ds'] - pd.DateOffset(weeks=3)
    alert_rows.append({
        'Week':          row['week_label'],
        'Technology':    row['Technology'],
        'Specialisation':row['Specialisation'],
        'Gap FTE':       round(row['gap_fte'], 1),
        'Gap Hours':     round(row['gap_hours'], 0),
        'Severity':      str(row['status']),
        'Role to Request': GRADE_HINTS.get(row['Specialisation'], 'See spec'),
        'Engage By':     engage_by.strftime('%d %b %Y'),
    })

alerts_df = pd.DataFrame(alert_rows)
if len(alerts_df):
    print(f'Q6: {len(alerts_df)} resource requirement alerts:\n')
    print(alerts_df.to_string(index=False))
else:
    print('No capacity breaches forecast — all technologies within target utilisation.')

---
## Q7 — Minimum Weekly Inputs to Refresh the Forecast

The forecast is fully automated from the two source files. The weekly task is a **30-minute process** with four inputs.

In [ ]:
# ── Q7: Weekly refresh function ────────────────────────────────────────────────
# This is the single function you call each week after updating the 4 inputs.

def weekly_refresh(
    timesheets_path:  str,             # 1. Updated time_work.xlsx export
    tickets_path:     str,             # 2. Updated assign.xlsx export
    demand_overlay:   pd.DataFrame,    # 3. Your demand overlay table (edit in cell above)
    headcount_dict:   dict,            # 4. Current headcount per technology
    utilisation_rate: float = 0.85,    # editable
    forecast_weeks:   int   = 13,
):
    """
    Full forecast refresh pipeline. Call once per week.

    Returns
    -------
    forecast_df   : pd.DataFrame   — weekly hours/FTE by tech × spec
    gap_df        : pd.DataFrame   — demand vs capacity gap
    alert_df      : pd.DataFrame   — resource request alerts (gap ≥ 0.5 FTE)
    """
    import logging
    logging.getLogger('prophet').setLevel(logging.WARNING)
    logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

    # 1. Reload source data
    xl_ts  = pd.ExcelFile(timesheets_path)
    ts_new = pd.concat([xl_ts.parse(s, parse_dates=['Date']) for s in xl_ts.sheet_names], ignore_index=True)
    ts_new['week_start'] = (ts_new['Date'] - pd.to_timedelta(ts_new['Date'].dt.dayofweek, unit='D')).dt.normalize()
    ts_new['ticket_type'] = ts_new['Number'].str[:3].fillna('UNK')
    ts_new['category']    = ts_new.apply(classify_row, axis=1)

    ts_del = ts_new[ts_new['category'].isin(['Delivery','Proactive Delivery'])].copy()

    # Fill null specs
    modal = (ts_del[ts_del['Specialisation'].notna()]
             .groupby('User')['Specialisation']
             .agg(lambda x: x.mode()[0] if len(x) else np.nan))
    ts_del['Specialisation'] = ts_del.apply(
        lambda r: modal.get(r['User'], r['Specialisation']) if pd.isna(r['Specialisation']) else r['Specialisation'],
        axis=1
    )

    # 2. Build weekly aggregate
    wkly = (ts_del.groupby(['week_start','Technology','Specialisation'])['Time worked']
            .sum().reset_index()
            .rename(columns={'week_start':'ds','Time worked':'y'})
            .query('y > 0'))

    # 3. Retrain Prophet on each valid cell
    hols_df = build_uk_holidays_df()
    fc_records = []
    start = pd.Timestamp.now().normalize() + pd.offsets.Week(weekday=0)
    cell_counts = wkly.groupby(['Technology','Specialisation']).size()
    valid = cell_counts[cell_counts >= 26].reset_index()[['Technology','Specialisation']]

    for _, vrow in valid.iterrows():
        cell_df = wkly[(wkly['Technology']==vrow['Technology']) & (wkly['Specialisation']==vrow['Specialisation'])]
        m, fc = train_cell_prophet(cell_df, hols_df)
        if fc is None: continue
        fc_fwd = fc[fc['ds'] >= start].head(forecast_weeks).copy()
        fc_fwd['Technology']    = vrow['Technology']
        fc_fwd['Specialisation'] = vrow['Specialisation']
        fc_fwd['forecast_hours'] = fc_fwd['yhat'].clip(0).round(1)
        fc_fwd['week_num']       = range(1, len(fc_fwd)+1)
        fc_records.append(fc_fwd)

    forecast_df = pd.concat(fc_records, ignore_index=True) if fc_records else pd.DataFrame()

    # 4. Merge demand overlay
    if not demand_overlay.empty:
        ov_agg = (demand_overlay
                  .groupby(['week_start','technology','specialisation'])['hours']
                  .sum().reset_index()
                  .rename(columns={'week_start':'ds','technology':'Technology',
                                   'specialisation':'Specialisation','hours':'overlay_hours'}))
        forecast_df = forecast_df.merge(ov_agg, on=['ds','Technology','Specialisation'], how='left')
        forecast_df['overlay_hours'] = forecast_df.get('overlay_hours', pd.Series(dtype=float)).fillna(0)
    else:
        forecast_df['overlay_hours'] = 0

    fte_div = HOURS_PER_WEEK * utilisation_rate
    forecast_df['total_hours'] = forecast_df['forecast_hours'] + forecast_df['overlay_hours']
    forecast_df['total_fte']   = (forecast_df['total_hours'] / fte_div).round(2)

    # 5. Compute gaps
    forecast_df['capacity_hours'] = forecast_df['Technology'].map(headcount_dict).fillna(0) * HOURS_PER_WEEK * utilisation_rate
    forecast_df['gap_hours']      = (forecast_df['total_hours'] - forecast_df['capacity_hours']).round(1)
    forecast_df['gap_fte']        = (forecast_df['gap_hours'] / fte_div).round(2)
    forecast_df['gap_pct']        = (forecast_df['gap_hours'] / forecast_df['capacity_hours'].replace(0, np.nan) * 100).round(1)

    # 6. Alerts
    alert_df = forecast_df[forecast_df['gap_fte'] >= 0.5].copy()
    alert_df['engage_by'] = pd.to_datetime(alert_df['ds']) - pd.DateOffset(weeks=3)

    return forecast_df, alert_df


print('weekly_refresh() defined.  Call it each Monday with updated file paths.')
print()
print('Example:')
print("  fc, alerts = weekly_refresh('time_work.xlsx', 'assign.xlsx', demand_overlay, headcount)")

In [ ]:
# ── Minimum weekly input checklist ────────────────────────────────────────────
checklist = pd.DataFrame([
    dict(input_no=1, what='Updated time_work.xlsx',  how='Export from PSA/ITSM — same format, same columns',  effort='Automated (scheduled export)',  mandatory=True),
    dict(input_no=2, what='Updated assign.xlsx',     how='Export from ServiceNow or ITSM — same format',      effort='Automated (scheduled export)',  mandatory=True),
    dict(input_no=3, what='Demand overlay updates',  how='Add/remove rows in the demand_overlay DataFrame above', effort='5–15 min manual',            mandatory=False),
    dict(input_no=4, what='Headcount changes',       how='Update headcount dict if anyone joins/leaves/moves',  effort='As-needed (< 2 min)',          mandatory=False),
])
print('Q7: Minimum weekly inputs to refresh the forecast:\n')
print(checklist.to_string(index=False))
print()
print('Total weekly effort: 5–30 minutes (zero if demand/headcount unchanged)')

---
## Q8 — Base, High, and Low Scenarios

In [ ]:
# ── Q8: Generate all three scenarios ──────────────────────────────────────────
# Scenario multipliers are applied to the BAU Prophet yhat (not overlay)
SCENARIO_CONFIGS = {
    'Low (P20)':    {'mult': 0.80, 'color': GREEN, 'linestyle': ':'},
    'Base (Median)':{'mult': 1.00, 'color': TEAL,  'linestyle': '-'},
    'High (P80)':   {'mult': 1.22, 'color': RED,   'linestyle': '--'},
}

scenario_results = {}

for scenario_name, cfg in SCENARIO_CONFIGS.items():
    sc_df = q1_with_overlay.copy()
    sc_df['scenario_hours'] = sc_df['forecast_hours'] * cfg['mult'] + sc_df['overlay_hours']
    sc_df['scenario_fte']   = (sc_df['scenario_hours'] / FTE_DIVISOR).round(2)
    sc_df['capacity_hours'] = sc_df['Technology'].map(calc_capacity)
    sc_df['gap_hours']      = sc_df['scenario_hours'] - sc_df['capacity_hours']
    sc_df['gap_pct']        = (sc_df['gap_hours'] / sc_df['capacity_hours'] * 100).round(1)
    sc_df['scenario']       = scenario_name
    scenario_results[scenario_name] = sc_df


def scenario_summary(df):
    weekly_totals = df.groupby('week_num')['scenario_hours'].sum()
    weekly_fte    = df.groupby('week_num')['scenario_fte'].sum()
    weekly_gap    = df.groupby('week_num')['gap_hours'].sum()
    return {
        'total_hours_13wk':   round(weekly_totals.sum()),
        'peak_fte_single_wk': round(weekly_fte.max(), 1),
        'avg_weekly_hours':   round(weekly_totals.mean()),
        'weeks_over_capacity': (weekly_gap > 0).sum(),
        'max_gap_hours':       round(weekly_gap.max()),
        'max_gap_fte':         round(weekly_gap.max() / FTE_DIVISOR, 1),
    }

summary_rows = []
for name, df in scenario_results.items():
    row = {'Scenario': name, **scenario_summary(df)}
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index('Scenario')
summary_df.columns = ['Total Hrs (13wk)', 'Peak FTE', 'Avg Hrs/wk', 'Wks Over Cap', 'Max Gap Hrs', 'Max Gap FTE']
print('Q8: Scenario comparison summary:\n')
print(summary_df.to_string())

In [ ]:
# ── Scenario chart ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Total weekly hours — all three scenarios
for scenario_name, cfg in SCENARIO_CONFIGS.items():
    df = scenario_results[scenario_name]
    wk_h = df.groupby('ds')['scenario_hours'].sum().sort_index()
    axes[0].plot(wk_h.index, wk_h.values,
                 color=cfg['color'], linestyle=cfg['linestyle'],
                 linewidth=2.0, label=scenario_name, alpha=0.9)

# Capacity line
total_cap = sum(v * HOURS_PER_WEEK * UTILISATION_RATE for v in headcount.values())
axes[0].axhline(total_cap, color=AMBER, linewidth=0.8, linestyle='-.', alpha=0.7, label=f'Total capacity ({total_cap:.0f}h)')
axes[0].set_title('Q8: Weekly total demand by scenario vs capacity', fontsize=11)
axes[0].set_ylabel('Total hours/week (all techs)')
axes[0].legend(fontsize=9, framealpha=0.1)
axes[0].grid(True, alpha=0.2)

# Gap (demand − capacity) — base only with confidence band
base_df   = scenario_results['Base (Median)'].groupby('ds')
high_df   = scenario_results['High (P80)'].groupby('ds')
low_df    = scenario_results['Low (P20)'].groupby('ds')

wks = sorted(scenario_results['Base (Median)']['ds'].unique())
base_gap = [scenario_results['Base (Median)'][scenario_results['Base (Median)']['ds']==w]['gap_hours'].sum() for w in wks]
high_gap = [scenario_results['High (P80)'][scenario_results['High (P80)']['ds']==w]['gap_hours'].sum() for w in wks]
low_gap  = [scenario_results['Low (P20)'][scenario_results['Low (P20)']['ds']==w]['gap_hours'].sum() for w in wks]

axes[1].fill_between(wks, low_gap, high_gap, alpha=0.12, color=TEAL, label='Low–High range')
axes[1].plot(wks, base_gap, color=TEAL, linewidth=2.0, label='Base gap')
axes[1].plot(wks, high_gap, color=RED,  linewidth=1.2, linestyle='--', alpha=0.7, label='High demand gap')
axes[1].plot(wks, low_gap,  color=GREEN,linewidth=1.2, linestyle=':',  alpha=0.7, label='Low demand gap')
axes[1].axhline(0, color='white', linewidth=0.8, alpha=0.4)
axes[1].fill_between(wks, 0, [max(0,g) for g in base_gap], alpha=0.15, color=RED)
axes[1].fill_between(wks, [min(0,g) for g in base_gap], 0, alpha=0.12, color=GREEN)
axes[1].set_title('Demand − Capacity gap by week (positive = over capacity)', fontsize=11)
axes[1].set_ylabel('Gap (hours)')
axes[1].set_xlabel('Week')
axes[1].legend(fontsize=9, framealpha=0.1)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ── Q8: Per-technology scenario breakdown ─────────────────────────────────────
fig, axes = plt.subplots(1, len(headcount), figsize=(16, 4), sharey=False)

for ax, (tech, hc) in zip(axes, headcount.items()):
    cap = hc * HOURS_PER_WEEK * UTILISATION_RATE
    totals = {}
    for sc_name, sc_df in scenario_results.items():
        sub = sc_df[sc_df['Technology'] == tech]
        totals[sc_name] = sub['scenario_hours'].sum() / FORECAST_WEEKS  # avg per week

    colors_bar = [GREEN, TEAL, RED]
    bars = ax.bar(range(3), list(totals.values()),
                  color=colors_bar, edgecolor='none', alpha=0.82, width=0.6)
    ax.axhline(cap, color=AMBER, linewidth=1.0, linestyle='--', alpha=0.8)
    ax.set_xticks(range(3))
    ax.set_xticklabels(['Low','Base','High'], fontsize=8)
    ax.set_title(tech.replace('Dynamics 365 ','D365\n').replace('Microsoft ','MS\n'), fontsize=8)
    ax.set_ylabel('Avg hrs/wk' if ax == axes[0] else '', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.grid(True, axis='y', alpha=0.2)

fig.suptitle(f'Q8: Average weekly demand by scenario vs capacity (amber = {UTILISATION_RATE*100:.0f}% util capacity)', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Full scenario × technology summary export ─────────────────────────────────
export_rows = []
for sc_name, sc_df in scenario_results.items():
    for tech, hc in headcount.items():
        sub = sc_df[sc_df['Technology'] == tech]
        if sub.empty: continue
        cap = hc * HOURS_PER_WEEK * UTILISATION_RATE
        export_rows.append({
            'Scenario':          sc_name,
            'Technology':        tech,
            'Headcount':         hc,
            'Weekly Capacity h': round(cap, 0),
            'Avg Demand h/wk':   round(sub['scenario_hours'].sum() / FORECAST_WEEKS, 0),
            'Total Hours 13wk':  round(sub['scenario_hours'].sum(), 0),
            'Peak FTE':          round(sub['scenario_fte'].sum(), 1),  # sum across specs
            'Weeks Over Cap':    (sub.groupby('week_num')['gap_hours'].sum() > 0).sum(),
        })

export_df = pd.DataFrame(export_rows)
print('Q8: Full scenario × technology output:\n')
print(export_df.to_string(index=False))

# Optional: export to Excel
# export_df.to_excel('q8_scenarios.xlsx', index=False)
# print('\nExported to q8_scenarios.xlsx')